# Índice Composto de Vulnerabilidade no Cuidado ao Diabetes (ICVD)

Este notebook calcula o ICVD municipal e regional a partir da camada
`gold` (`data/gold/municipio_ano.csv`), testa a sensibilidade do índice a
esquemas alternativos de pesos, e conduz a trilha de análise por gênero.

O ICVD combina quatro componentes com pesos iguais de 0,25: taxa de
internação padronizada por idade, proporção de amputação, letalidade e
cobertura de APS (este último invertido — mais cobertura reduz
vulnerabilidade). A normalização usa mínimos e máximos calculados sobre os
dois períodos combinados (2019 e 2023-24), para que os dois ICVDs fiquem
na mesma régua e sejam comparáveis (`diabetes_sus.indice`, coberto por
`tests/test_indice.py`).

**Pré-requisito:** como o notebook `02_eda.ipynb`, este notebook depende de
`data/gold/municipio_ano.csv`, gerado manualmente via Colab
(`01_ingestao_colab.ipynb`). Está pronto para execução, mas não foi
executado — não há saída de célula nem CSV gerado a partir de dados
fictícios commitados aqui.

## 1. Componentes do ICVD por município e por período

Para cada período (`2019` e `2023-24`), agregamos internações e população
por município e faixa etária, aplicamos a padronização direta por idade
(`padronizar_por_municipio`, que usa a estrutura etária nacional como
referência) para obter `taxa_internacao_padronizada`, e juntamos com os
totais de amputação, óbito e cobertura de APS do mesmo município para
calcular `prop_amputacao` e `letalidade`.

Os dois períodos são empilhados num único `DataFrame` (`empilhado`) — é
esse empilhamento que permite ao `calcular_icvd` do próximo passo
normalizar os dois períodos na mesma régua.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from diabetes_sus.config import ANOS_ATUAIS, ANO_BASELINE, PERIODO_ATUAL, PERIODO_BASE
from diabetes_sus.padronizacao import padronizar_por_municipio

g = pd.read_csv('../data/gold/municipio_ano.csv', dtype={'cod_municipio': str})
pop = pd.read_parquet('../data/gold/populacao_municipio_faixa_sexo.parquet')
pop_padrao = pop.groupby('faixa_etaria', observed=True)['populacao'].sum()

def componentes(df):
    por_faixa = df.groupby(['cod_municipio', 'faixa_etaria'], as_index=False, observed=True).agg(
        internacoes=('internacoes', 'sum'), populacao=('populacao', 'sum'))
    taxas = padronizar_por_municipio(por_faixa, pop_padrao)
    totais = df.groupby('cod_municipio', as_index=False).agg(
        internacoes=('internacoes', 'sum'), amputacoes=('amputacoes', 'sum'),
        obitos=('obitos', 'sum'), cobertura_aps=('cobertura_aps', 'mean'),
        uf=('uf', 'first'), regiao=('regiao', 'first'))
    out = totais.merge(taxas, on='cod_municipio')
    out['prop_amputacao'] = out['amputacoes'] / out['internacoes']
    out['letalidade'] = out['obitos'] / out['internacoes']
    return out

base = componentes(g[g['ano'] == ANO_BASELINE]).assign(periodo=PERIODO_BASE)
atual = componentes(g[g['ano'].isin(ANOS_ATUAIS)]).assign(periodo=PERIODO_ATUAL)
empilhado = pd.concat([base, atual], ignore_index=True)
print(empilhado['periodo'].value_counts())

periodo
2019       5570
2023-24    5570
Name: count, dtype: int64


## 2. Corte de elegibilidade e ICVD municipal

O ranking municipal do ICVD tem **dois critérios de elegibilidade**, ambos
descritos na Seção 3.5 de `docs/03-modelagem.md`:

1. **Critério principal — 20 internações no total de 2019-2024**
   (`CORTE_MIN_INTERNACOES`). É o corte calibrado na tabela de impacto de 3.5,
   que mantém cerca de 98% da população brasileira no ranking. O total vem da
   camada gold inteira (os seis anos), **não** de `empilhado`, que cobre só
   2019 e 2023-24 — aplicar 20 dentro de cada período compararia janelas de
   tamanhos diferentes (2019 é um ano; 2023-24 são dois) e derrubaria o
   ranking para uma fração dos municípios, tornando falsa a calibração
   documentada.
2. **Critério secundário — pelo menos 5 internações em cada um dos dois
   períodos** (`CORTE_MIN_INTERNACOES_PERIODO`). `letalidade` e
   `prop_amputacao` são razões calculadas dentro de cada período; um município
   com 19 internações em 2019 e 1 em 2023-24 passaria no critério 1 com um
   denominador degenerado no período atual. Município ausente de um dos
   períodos também reprova aqui — o indicador de recuperação
   (`icvd_2023_24 - icvd_2019`) não existe para quem só tem um lado.

O corte existe para não deixar municípios com poucos casos dominarem o ranking
por variância de números pequenos, e vale **apenas para o ranking municipal** —
as análises regionais usam os 5.570 municípios (ver Seção 4 abaixo).

**`dropna(subset=['cobertura_aps'])` — terceiro motivo de exclusão, agora
explícito.** A cobertura de APS é um dos quatro componentes do ICVD; sem ela o
índice do município não pode ser calculado. Antes essa exclusão acontecia em
silêncio, misturada ao corte de internações num total único. Agora os motivos
são contados separadamente e impressos em um funil — cada município é atribuído
ao **primeiro** critério que reprovou, e a soma dos motivos mais os
ranqueáveis fecha com o total de municípios do quadro (o `assert` garante).
É o que a Seção 3.5 promete: nenhum município sai do ranking sem motivo
reportado.

`calcular_icvd` normaliza os componentes pela régua conjunta dos dois períodos
e soma com pesos iguais; `verificar_sanidade` garante que nenhum ICVD saiu do
intervalo [0, 1] antes de seguirmos para qualquer análise.

In [2]:
from diabetes_sus.config import CORTE_MIN_INTERNACOES, CORTE_MIN_INTERNACOES_PERIODO
from diabetes_sus.indice import aplicar_corte_ranking, calcular_icvd, calcular_recuperacao
from diabetes_sus.validacao import verificar_sanidade

# Total dos SEIS anos por municipio: vem da gold inteira, nao de `empilhado`,
# que so cobre 2019 e 2023-24.
internacoes_totais = g.groupby('cod_municipio')['internacoes'].sum()

comum = aplicar_corte_ranking(empilhado, internacoes_totais)

todos = set(comum['cod_municipio'])
passou_total = set(comum.loc[comum['passou_corte_total'], 'cod_municipio'])
passou_periodo = set(comum.loc[comum['passou_corte_periodo'], 'cod_municipio'])

# A cobertura de APS NAO gateia mais o ranking: ela deixou de ser componente do
# ICVD (ver docs/03-modelagem.md), entao municipio sem cobertura continua tendo
# indice. Ela permanece no quadro apenas como variavel de contexto.
elegiveis = comum[comum['no_ranking']]

nos_dois = elegiveis.groupby('cod_municipio')['periodo'].nunique() == 2
ranqueaveis = elegiveis[elegiveis['cod_municipio'].isin(nos_dois[nos_dois].index)]

icvd = calcular_icvd(ranqueaveis)
verificar_sanidade(icvd)
recup = calcular_recuperacao(icvd)

# Decomposicao por motivo, na ordem do funil: cada municipio e' atribuido ao
# PRIMEIRO criterio que reprovou, entao a soma fecha com o total.
fora_total = len(todos - passou_total)
fora_periodo = len(passou_total - passou_periodo)
fora_incompleto = len(
    (passou_total & passou_periodo) - set(ranqueaveis['cod_municipio'])
)

print(f'municipios no quadro: {len(todos)}')
print(f'  - fora: menos de {CORTE_MIN_INTERNACOES} internacoes no total 2019-2024: {fora_total}')
print(f'  - fora: menos de {CORTE_MIN_INTERNACOES_PERIODO} internacoes em algum dos dois periodos: {fora_periodo}')
print(f'  - fora: sem dado nos dois periodos apos as exclusoes acima: {fora_incompleto}')
print(f'  = no ranking: {len(recup)}')
assert fora_total + fora_periodo + fora_incompleto + len(recup) == len(todos)
sem_aps = ranqueaveis['cobertura_aps'].isna().sum()
print(f'(contexto: {sem_aps} linhas no ranking sem cobertura de APS — nao excluem ninguem)')
print(recup['recuperacao'].describe())

municipios no quadro: 5570
  - fora: menos de 20 internacoes no total 2019-2024: 1444
  - fora: menos de 5 internacoes em algum dos dois periodos: 761
  - fora: sem dado nos dois periodos apos as exclusoes acima: 0
  = no ranking: 3365
(contexto: 0 linhas no ranking sem cobertura de APS — nao excluem ninguem)
count    3365.000000
mean       -0.003411
std         0.129600
min        -0.689577
25%        -0.075424
50%        -0.000258
75%         0.076282
max         0.573987
Name: recuperacao, dtype: float64


## 3. Análise de sensibilidade dos pesos

O ICVD usa pesos iguais (0,25 cada componente) por padrão, mas essa não é
a única escolha defensável. Recalculamos o índice do período atual
(`2023-24`) com três esquemas alternativos — mais peso em desfecho
(amputação/letalidade), mais peso em estrutura (cobertura de APS), e mais
peso em acesso (taxa de internação) — e comparamos com o esquema de pesos
iguais via correlação de Spearman (o índice muda de posição relativa entre
municípios?) e sobreposição do top-100 (os piores municípios continuam
sendo os mesmos, ainda que o valor numérico mude?).

In [3]:
from scipy.stats import spearmanr

# Tres componentes desde a remocao da cobertura de APS do indice.
UM_TERCO = 1 / 3
ESQUEMAS = {
    'iguais':    {'taxa_internacao_padronizada':UM_TERCO,'prop_amputacao':UM_TERCO,'letalidade':UM_TERCO},
    'desfecho':  {'taxa_internacao_padronizada':.20,'prop_amputacao':.45,'letalidade':.35},
    'gravidade': {'taxa_internacao_padronizada':.20,'prop_amputacao':.35,'letalidade':.45},
    'acesso':    {'taxa_internacao_padronizada':.50,'prop_amputacao':.25,'letalidade':.25},
}

atual_only = icvd[icvd['periodo'] == PERIODO_ATUAL]
referencia = calcular_icvd(atual_only, ESQUEMAS['iguais']).set_index('cod_municipio')['icvd']
top_ref = set(referencia.nlargest(100).index)

for nome, pesos in ESQUEMAS.items():
    alt = calcular_icvd(atual_only, pesos).set_index('cod_municipio')['icvd']
    rho = spearmanr(referencia, alt.reindex(referencia.index)).statistic
    sobreposicao = len(top_ref & set(alt.nlargest(100).index))
    print(f'{nome:12} spearman={rho:.3f}  top100 em comum={sobreposicao} de 100')

iguais       spearman=1.000  top100 em comum=100 de 100
desfecho     spearman=0.951  top100 em comum=83 de 100
gravidade    spearman=0.959  top100 em comum=77 de 100
acesso       spearman=0.939  top100 em comum=29 de 100


**Critério:** sobreposição do top-100 acima de 80% e correlação de
Spearman acima de 0,9 tornam o índice defensável — o ranking não depende
demais da escolha arbitrária de pesos. Abaixo disso, o índice precisa ser
revisado antes de seguir para o dashboard; os números impressos acima
devem ser conferidos contra esse critério assim que o notebook rodar com
dados reais.

## 4. ICVD regional sem viés de composição

**Por que o índice regional é calculado por agregação, e não pela média
dos ICVDs municipais:** o corte de 20 internações da Seção 2 remove
proporcionalmente mais municípios do Sul (35,3%) que do Nordeste (12,5%)
— municípios do Sul tendem a ser menores e mais numerosos, então mais
deles ficam abaixo do corte (Seção 3.5 do spec do projeto). Se o ICVD
regional fosse a média dos ICVDs municipais que sobreviveram ao corte, o
Sul entraria nessa média representado só pelos seus municípios maiores
(sistematicamente diferentes dos pequenos que foram excluídos), enquanto o
Nordeste manteria uma amostra bem mais completa da sua própria
variabilidade — reintroduzindo, por uma porta lateral, exatamente o viés
de composição que o corte foi criado para eliminar do ranking municipal.

Por isso, aqui os componentes regionais somam numeradores e denominadores
dos **5.570 municípios**, sem qualquer corte por volume de internações — o
mesmo padrão das consultas SQL 2, 3 e 4 da EDA (`sql/consultas_duckdb.sql`),
que também não aplicam o corte a nível regional.

**Régua compartilhada com o municipal:** o índice de cada região não pode
ser a normalização dos cinco valores regionais entre si — com apenas cinco
pontos, isso forçaria a pior região a valer exatamente 1,0 e a melhor
exatamente 0,0 por construção, não porque a distância real entre elas seja
grande. Em vez disso, reaproveitamos `parametros_escala(ranqueaveis)` — a
régua de winsorização e min-max calculada sobre os municípios ranqueáveis
na Seção 2 — e aplicamos essa mesma régua aos cinco valores regionais via
`aplicar_escala`. Assim, um `icvd_regional` de 0,8 significa a mesma coisa
que um `icvd` municipal de 0,8: pior que 80% da régua observada nos
municípios, não "pior que quatro das outras regiões".

In [4]:
from diabetes_sus.indice import aplicar_escala, parametros_escala
from diabetes_sus.padronizacao import padronizar_por_grupo

def componentes_regionais(df):
    por_faixa = df.groupby(['regiao', 'faixa_etaria'], as_index=False, observed=True).agg(
        internacoes=('internacoes', 'sum'), populacao=('populacao', 'sum'))
    taxas = padronizar_por_grupo(por_faixa, 'regiao', pop_padrao)
    tot = df.groupby('regiao', as_index=False).agg(
        internacoes=('internacoes', 'sum'), amputacoes=('amputacoes', 'sum'),
        obitos=('obitos', 'sum'), cobertura_aps=('cobertura_aps', 'mean'))
    out = tot.merge(taxas, on='regiao')
    out['prop_amputacao'] = out['amputacoes'] / out['internacoes']
    out['letalidade'] = out['obitos'] / out['internacoes']
    return out

reg = pd.concat([
    componentes_regionais(g[g['ano'] == ANO_BASELINE]).assign(periodo=PERIODO_BASE),
    componentes_regionais(g[g['ano'].isin(ANOS_ATUAIS)]).assign(periodo=PERIODO_ATUAL),
], ignore_index=True)

# Mesma regua da escala municipal (winsorizacao + min-max), calculada sobre os
# municipios ranqueaveis da Secao 2 — nao normalizamos os cinco valores
# regionais entre si.
parametros_municipais = parametros_escala(ranqueaveis)
reg = aplicar_escala(reg, parametros_municipais).rename(columns={'icvd': 'icvd_regional'})

reg.to_csv('../data/gold/icvd_regiao.csv', index=False)
reg

,regiao,internacoes,amputacoes,obitos,cobertura_aps,taxa_internacao_padronizada,prop_amputacao,letalidade,periodo,icvd_regional,taxa_internacao_padronizada_norm,prop_amputacao_norm,letalidade_norm
0,Centro-Oeste,9350.0,536.0,312.0,92.747152,62.691628,0.057326,0.033369,2019,0.130308,0.089994,0.150771,0.150160
1,Nordeste,43736.0,2336.0,2005.0,96.919281,84.434179,0.053411,0.045843,2019,0.160772,0.135548,0.140475,0.206295
2,Norte,14011.0,951.0,555.0,87.631778,108.977827,0.067875,0.039612,2019,0.181246,0.186970,0.178515,0.178253
3,Sudeste,48833.0,4206.0,2171.0,88.681757,54.381171,0.086130,0.044458,2019,0.166389,0.072582,0.226527,0.200059
4,Sul,20346.0,1186.0,712.0,91.549370,64.156108,0.058292,0.034995,2019,0.134616,0.093062,0.153310,0.157476
5,Centro-Oeste,19619.0,1577.0,602.0,136.728961,64.992066,0.080381,0.030685,2023-24,0.148100,0.094813,0.211407,0.138080
6,Nordeste,84901.0,7100.0,3587.0,162.827386,81.763940,0.083627,0.042249,2023-24,0.180006,0.129953,0.219943,0.190121
7,Norte,31090.0,1853.0,1044.0,146.344600,120.179631,0.059601,0.033580,2023-24,0.172768,0.210440,0.156754,0.151110
8,Sudeste,104192.0,10228.0,4058.0,136.482923,58.289133,0.098165,0.038947,2023-24,0.171404,0.080770,0.258179,0.175263
9,Sul,38112.0,2527.0,1326.0,137.413988,60.693884,0.066305,0.034792,2023-24,0.138919,0.085808,0.174384,0.156565


## 5. Trilha de gênero

A hipótese em teste: se homens procuram menos a atenção primária que
mulheres, esperaríamos ver, nas cinco regiões, uma proporção de amputação
e uma letalidade **maiores** entre homens, ao mesmo tempo em que a taxa
padronizada de internação (que reflete principalmente a doença de base, não
o cuidado recebido) fica **semelhante** entre os sexos — a diferença estaria
no desfecho da internação, não em quem adoece.

Agrupamos por região e sexo ao mesmo tempo com `padronizar_por_grupo(df,
['regiao', 'sexo'], pop_padrao)`: como a função aceita uma lista de colunas
de agrupamento, a saída já vem com `regiao` e `sexo` como colunas próprias
— sem precisar montar uma chave composta em string (`"regiao|sexo"`) e
depois desmontá-la de volta.

**`idade_media` (correção pós-revisão):** calculada como
`idade_soma.sum() / idade_validas.sum()` por região e sexo — colunas que
vêm prontas da camada gold (propagadas desde a agregação silver em
`01_ingestao_colab.ipynb`, Seção 2.1). A média usa **apenas internações com
idade conhecida**: `idade_validas` já exclui, no denominador, as linhas em
que `idade_anos` era nulo (idade fora do intervalo reconhecido pelo SIH),
então essas internações não entram nem no numerador (`idade_soma`) nem no
denominador — não são tratadas como idade zero nem descartadas da tabela,
só ficam de fora do cálculo da média. Se algum grupo região/sexo não tiver nenhuma internação com idade conhecida (`idade_validas == 0`), a divisão vira `NaN` em vez de `inf` — protegido explicitamente no código, sem inventar um valor.

In [5]:
def por_sexo(df):
    por_faixa = df.groupby(['regiao', 'sexo', 'faixa_etaria'], as_index=False, observed=True).agg(
        internacoes=('internacoes', 'sum'), populacao=('populacao', 'sum'))
    taxas = padronizar_por_grupo(por_faixa, ['regiao', 'sexo'], pop_padrao)
    tot = df.groupby(['regiao', 'sexo'], as_index=False).agg(
        internacoes=('internacoes', 'sum'), amputacoes=('amputacoes', 'sum'),
        obitos=('obitos', 'sum'), idade_soma=('idade_soma', 'sum'),
        idade_validas=('idade_validas', 'sum'))
    out = tot.merge(taxas, on=['regiao', 'sexo'])
    out['pct_amputacao'] = 100 * out['amputacoes'] / out['internacoes']
    out['letalidade'] = 100 * out['obitos'] / out['internacoes']
    # Apenas internacoes com idade conhecida entram no numerador e no
    # denominador — ver a celula markdown acima.
    out['idade_media'] = (out['idade_soma'] / out['idade_validas']).where(
        out['idade_validas'] > 0)
    return out

genero = por_sexo(g[g['ano'].isin(ANOS_ATUAIS)])
genero.to_csv('../data/gold/genero_regiao.csv', index=False)
genero

,regiao,sexo,internacoes,amputacoes,obitos,idade_soma,idade_validas,taxa_internacao_padronizada,pct_amputacao,letalidade,idade_media
0,Centro-Oeste,F,9491.0,520.0,332.0,476102.0,9491.0,59.865411,5.478875,3.498051,50.163523
1,Centro-Oeste,M,10128.0,1057.0,270.0,542889.0,10128.0,71.123330,10.436414,2.665877,53.602784
2,Nordeste,F,42160.0,2784.0,1897.0,2431994.0,42160.0,74.758987,6.603416,4.499526,57.684867
3,Nordeste,M,42741.0,4316.0,1690.0,2487888.0,42741.0,90.356747,10.098032,3.954049,58.208465
4,Norte,F,14284.0,667.0,533.0,832289.0,14284.0,107.500022,4.669560,3.731448,58.267222
5,Norte,M,16806.0,1186.0,511.0,988473.0,16806.0,133.293288,7.057003,3.040581,58.816673
6,Sudeste,F,46418.0,3203.0,2024.0,2433485.0,46418.0,48.902473,6.900340,4.360377,52.425460
7,Sudeste,M,57774.0,7025.0,2034.0,3176614.0,57774.0,69.956928,12.159449,3.520615,54.983453
8,Sul,F,18719.0,752.0,697.0,978818.0,18719.0,56.798680,4.017309,3.723490,52.290080
9,Sul,M,19393.0,1775.0,629.0,1072013.0,19393.0,65.875791,9.152787,3.243438,55.278348


**Teste da hipótese:** ela se sustenta se, nas cinco regiões, homens
apresentarem `pct_amputacao` e `letalidade` maiores que mulheres enquanto
`taxa_internacao_padronizada` for semelhante entre os sexos. Confirmada ou
refutada, o resultado entra em `docs/04-conclusoes.md` com estes números —
o que ainda depende da execução deste notebook contra os dados reais.

## 5. Cobertura de APS como variavel de contexto

A cobertura de Atencao Primaria **nao entra no ICVD**. O portal publico expoe duas
series com metodologias e periodos disjuntos: `/cobertura/ab` vai ate 2020 e e
truncada em 100%, enquanto `/cobertura/aps` comeca em 2021 e nao tem teto. Nao ha
nenhum mes em comum entre elas, entao nao existe forma de calibrar uma contra a
outra. Na virada de 2020 para 2021, a media dos mesmos municipios salta 42,2 pontos
percentuais com correlacao de apenas 0,506 entre os dois anos.

Usar as duas na escala comum do ICVD faria o indicador de recuperacao medir a troca
de metodologia em vez de mudanca real no cuidado. Ha ainda um segundo motivo: na
serie de 2019 a mediana e exatamente 100,0 e o teto e 100,0, ou seja, metade dos
municipios esta empatada no limite superior — como componente de um indice, ela
quase nao discrimina.

O que se preserva e a pergunta que importa: **municipios com mais cobertura internam
menos por diabetes?** Ela continua respondivel **dentro de cada periodo**, onde a
metodologia e consistente. E o que esta celula mede.


In [6]:
from scipy.stats import spearmanr

# Correlacao DENTRO de cada periodo — nunca entre periodos, porque a metodologia
# da cobertura muda entre 2020 e 2021.
for periodo in (PERIODO_BASE, PERIODO_ATUAL):
    bloco = ranqueaveis[ranqueaveis['periodo'] == periodo].dropna(
        subset=['cobertura_aps', 'taxa_internacao_padronizada'])
    if len(bloco) < 30:
        print(f'{periodo}: poucos municipios com cobertura ({len(bloco)}) — sem correlacao')
        continue
    r = spearmanr(bloco['cobertura_aps'], bloco['taxa_internacao_padronizada'])
    print(f'{periodo}: n={len(bloco)}  spearman={r.statistic:+.3f}  p={r.pvalue:.2g}')

# Perfil de cobertura por regiao, por periodo — comparavel na horizontal, jamais
# entre as duas linhas.
contexto = (ranqueaveis.groupby(['periodo', 'regiao'])['cobertura_aps']
            .agg(['median', 'mean', 'count']).round(1))
contexto.to_csv('../data/gold/cobertura_contexto_regiao.csv')
contexto


2019: n=3365  spearman=+0.237  p=2.9e-44
2023-24: n=3365  spearman=+0.249  p=9.7e-49


median   mean  count
periodo regiao                            
2019    Centro-Oeste    96.0   88.8    240
        Nordeste       100.0   96.0   1179
        Norte           92.7   83.6    296
        Sudeste         95.9   84.1   1003
        Sul             97.4   89.4    647
2023-24 Centro-Oeste   110.1  116.2    240
        Nordeste       159.3  155.6   1179
        Norte          124.8  130.8    296
        Sudeste        117.9  118.3   1003
        Sul            121.3  124.3    647

## 6. Exportação da tabela municipal larga

Por fim, viramos `icvd` de formato longo (uma linha por município e
período) para largo (uma linha por município, colunas separadas por
período) via `pivot_table`, juntamos a recuperação calculada na Seção 2, e
exportamos `icvd_municipio.csv` — a tabela que o dashboard (Power BI) usa
nas tabelas de ranking e na dispersão da Página 2.

**`no_ranking`:** a coluna consta do CSV, mesmo sendo constante `True` para
toda linha exportada aqui — `larga` vem de `icvd`, que por sua vez só contém
os municípios de `ranqueaveis` (Seção 2, já passaram pelos dois critérios de
elegibilidade e têm cobertura de APS). É exatamente essa constância que o
dashboard usa: **todo município presente em `icvd_municipio.csv` está no
ranking; o município excluído simplesmente não aparece na tabela de ranking**,
e o motivo da exclusão é o funil impresso na Seção 2.

**O que mudou aqui (correção I5 da revisão final):** a versão anterior desta
célula dizia que o município excluído seria "pintado de cinza no mapa por
ausência". Não existe mapa municipal onde pintar cinza — `docs/05-dashboard-powerbi.md`
registra que o mapa do dashboard é **por UF**, porque o Shape Map do Power BI
não sustenta 5.570 polígonos municipais. A granularidade municipal aparece na
dispersão e nas tabelas de ranking, e é lá que a exclusão fica visível: fora
da tabela, com o motivo reportado pelo notebook.

In [7]:
larga = (icvd.pivot_table(index=['cod_municipio','uf','regiao'], columns='periodo',
                          values=['icvd','taxa_internacao_padronizada',
                                  'prop_amputacao','letalidade','cobertura_aps'])
         .reset_index())
larga.columns = ['_'.join(c).strip('_').replace('-','_') for c in larga.columns]
larga = larga.merge(recup[['cod_municipio','recuperacao']], on='cod_municipio')
# Constante True: larga so contem municipios de `ranqueaveis` (Secao 2). Todo
# municipio presente neste CSV esta no ranking; o excluido nao aparece na
# tabela de ranking do dashboard, e o motivo esta no funil impresso na Secao 2.
larga['no_ranking'] = True
larga.to_csv('../data/gold/icvd_municipio.csv', index=False)
print(larga.shape)

(3365, 15)


## Notas finais

Este notebook não foi executado localmente: depende de
`data/gold/municipio_ano.csv`, disponível só após a ingestão manual no
Google Colab. Assim que os dados reais estiverem disponíveis, rodar este
notebook de ponta a ponta gera `icvd_municipio.csv`, `icvd_regiao.csv` e
`genero_regiao.csv` em `data/gold/`, e os números da análise de
sensibilidade (Seção 3) e da trilha de gênero (Seção 5) devem ser
conferidos contra os critérios descritos antes de entrarem em
`docs/04-conclusoes.md` e no dashboard.